# Aerial GCP Pose Estimation using YOLO

## Objective
Detect Ground Control Point (GCP) markers and estimate their center locations in aerial imagery.

## Workflow
1. Configuration
2. Dataset Inspection
3. Exploratory Data Analysis (EDA)
4. Data Cleaning
5. YOLO Dataset Generation
6. Model Training
7. Inference
8. Prediction Export




In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CONFIG = {
    "TRAIN_ROOT": "/content/drive/MyDrive/GCP_Assignment_Datasets/train_dataset",
    "TEST_ROOT": "/content/drive/MyDrive/GCP_Assignment_Datasets/test_dataset",
    "JSON_PATH": "/content/drive/MyDrive/GCP_Assignment_Datasets/train_dataset/gcp_marks.json",

    "YOLO_ROOT": "/content/gcp_yolo",
    "PREDICTIONS_JSON": "predictions.json",

     "MODEL_PATH": "/content/drive/MyDrive/runs/detect/train-5/weights/best.pt",


    "BOX_SIZE": 48,
    "IMAGE_SIZE": 1536,
    "BATCH_SIZE": 8,
    "EPOCHS": 100,
    "YOLO_MODEL": "yolo11s.pt",
    "CONF_THRESHOLD" : 0.25,

    "RANDOM_SEED": 42,
}


In [ ]:
!pip install ultralytics

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

import os
import json
import random
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

In [ ]:
# ============================================================
# LOAD DATASET
# ============================================================

with open(CONFIG["JSON_PATH"], "r") as f:
    labels = json.load(f)

print(f"Total annotations: {len(labels)}")

In [ ]:
print("Total samples:", len(labels))

In [ ]:
shape_counter = Counter()

for item in labels.values():
    if "verified_shape" in item:
        shape_counter[item["verified_shape"]] += 1

shape_counter

In [ ]:
bad_samples = []

for path, data in labels.items():

    if "verified_shape" not in data:
        bad_samples.append(path)

print("Invalid samples:", len(bad_samples))

In [ ]:
xs = [v["mark"]["x"] for v in labels.values()]
ys = [v["mark"]["y"] for v in labels.values()]

plt.figure(figsize=(6,6))
plt.scatter(xs, ys, alpha=0.3)
plt.title("Marker Distribution")
plt.show()

In [ ]:
clean_labels = {
    k: v
    for k, v in labels.items()
    if "verified_shape" in v
}

In [ ]:
clean_labels = {}

for k, v in labels.items():

    if "verified_shape" not in v:
        continue

    if "mark" not in v:
        continue

    if "x" not in v["mark"] or "y" not in v["mark"]:
        continue

    clean_labels[k] = v

labels = clean_labels

print("Usable samples:", len(labels))

In [ ]:


# ============================================================
# IMAGE LOOKUP
# ============================================================

ROOT = CONFIG["TRAIN_ROOT"]

image_lookup = {}

for root, _, files in os.walk(ROOT):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")):
            image_lookup[f] = os.path.join(root, f)

print(f"Images found: {len(image_lookup)}")

In [ ]:


ROOT = CONFIG["TRAIN_ROOT"]

image_lookup = {}

for root, dirs, files in os.walk(ROOT):

    for f in files:

        if not f.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        full_path = os.path.join(root, f)

        rel_path = os.path.relpath(full_path, ROOT)

        image_lookup[rel_path] = full_path

print("Indexed:", len(image_lookup))

In [ ]:
image_lookup = {}

for root, dirs, files in os.walk(ROOT):

    gcp_id = os.path.basename(root)

    for f in files:

        if f.lower().endswith((".jpg", ".jpeg", ".png")):

            image_lookup[(gcp_id, f)] = os.path.join(root, f)

In [ ]:
matched = 0
unmatched = 0

for json_path in labels.keys():

    parts = json_path.split("/")

    gcp_id = parts[-2]
    filename = parts[-1]

    if (gcp_id, filename) in image_lookup:
        matched += 1
    else:
        unmatched += 1

print("Matched:", matched)
print("Unmatched:", unmatched)

In [ ]:


samples = random.sample(list(labels.keys()), 25)

fig, axs = plt.subplots(5, 5, figsize=(15,15))

for ax, json_path in zip(axs.ravel(), samples):

    parts = json_path.split("/")

    gcp_id = parts[-2]
    filename = parts[-1]

    img_path = image_lookup[(gcp_id, filename)]

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    x = int(labels[json_path]["mark"]["x"])
    y = int(labels[json_path]["mark"]["y"])

    crop_size = 128

    y1 = max(0, y-crop_size)
    y2 = min(img.shape[0], y+crop_size)

    x1 = max(0, x-crop_size)
    x2 = min(img.shape[1], x+crop_size)

    crop = img[y1:y2, x1:x2]

    ax.imshow(crop)
    ax.axis("off")
    ax.set_title(labels[json_path]["verified_shape"], fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:


for json_path in random.sample(list(labels.keys()), 10):

    parts = json_path.split("/")

    gcp_id = parts[-2]
    filename = parts[-1]

    img_path = image_lookup[(gcp_id, filename)]

    img = cv2.imread(img_path)

    h, w = img.shape[:2]

    print(w, h)

In [ ]:


xs = [v["mark"]["x"] for v in labels.values()]
ys = [v["mark"]["y"] for v in labels.values()]

plt.figure(figsize=(8,6))
plt.scatter(xs, ys, s=5)

plt.xlabel("X")
plt.ylabel("Y")
plt.title("GCP Locations")

plt.gca().invert_yaxis()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import random

sizes = []

for path in random.sample(list(labels.keys()),100):

    parts = path.split("/")

    gcp_id = parts[-2]
    filename = parts[-1]

    img_path = image_lookup[(gcp_id,filename)]

    img = cv2.imread(img_path)

    x = int(labels[path]["mark"]["x"])
    y = int(labels[path]["mark"]["y"])

    crop = img[y-64:y+64,x-64:x+64]

    gray = cv2.cvtColor(crop,cv2.COLOR_BGR2GRAY)

    _,th = cv2.threshold(gray,180,255,cv2.THRESH_BINARY)

    contours,_ = cv2.findContours(
        th,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours):

        c = max(contours,key=cv2.contourArea)

        _,_,w,h = cv2.boundingRect(c)

        sizes.append(max(w,h))

plt.hist(sizes,bins=20)
plt.title("Marker Size Distribution")
plt.show()

print(np.mean(sizes))

In [ ]:
from sklearn.model_selection import train_test_split

all_paths = list(labels.keys())

train_paths, val_paths = train_test_split(
    all_paths,
    test_size=0.2,
    random_state=42,
    stratify=[labels[p]["verified_shape"] for p in all_paths]
)

In [ ]:


ROOT_test = CONFIG["TEST_ROOT"]

imagelookup = {}

for root, dirs, files in os.walk(ROOT_test):

    for f in files:

        if f.lower().endswith((".jpg", ".jpeg", ".png")):

            imagelookup[f] = os.path.join(ROOT_test, f)

print("Images found:", len(imagelookup))

In [ ]:
from sklearn.model_selection import train_test_split

all_paths = list(labels.keys())

train_paths, val_paths = train_test_split(
    all_paths,
    test_size=0.2,
    random_state=42,
    stratify=[labels[p]["verified_shape"] for p in all_paths]
)

print("Train:", len(train_paths))
print("Val:", len(val_paths))

In [ ]:


YOLO_ROOT = CONFIG["YOLO_ROOT"]

dirs = [
    "images/train",
    "images/val",
    "labels/train",
    "labels/val"
]

for d in dirs:
    os.makedirs(os.path.join(YOLO_ROOT, d), exist_ok=True)

In [ ]:
CLASS_MAP = {
    "Cross": 0,
    "Square": 1,
    "L-Shape": 2
}

In [ ]:


def process_split(split_paths, split_name):

    for json_path in tqdm(split_paths):

        parts = json_path.split("/")

        gcp_id = parts[-2]
        filename = parts[-1]

        img_path = image_lookup[(gcp_id, filename)]

        img = cv2.imread(img_path)

        h, w = img.shape[:2]

        x = labels[json_path]["mark"]["x"]
        y = labels[json_path]["mark"]["y"]

        shape = labels[json_path]["verified_shape"]

        cls = CLASS_MAP[shape]

        x_center = x / w
        y_center = y / h

        BOX_SIZE = CONFIG['BATCH_SIZE']

        box_w = BOX_SIZE / w
        box_h = BOX_SIZE / h

        label_line = (
            f"{cls} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{box_w:.6f} "
            f"{box_h:.6f}\n"
        )

        dst_img = os.path.join(
            YOLO_ROOT,
            f"images/{split_name}",
            filename
        )

        dst_lbl = os.path.join(
            YOLO_ROOT,
            f"labels/{split_name}",
            filename.replace(".JPG", ".txt")
                    .replace(".jpg", ".txt")
                    .replace(".jpeg", ".txt")
        )

        shutil.copy2(img_path, dst_img)

        with open(dst_lbl, "w") as f:
            f.write(label_line)

process_split(train_paths, "train")
process_split(val_paths, "val")

In [ ]:

sample = random.choice(train_paths)

parts = sample.split("/")

gcp_id = parts[-2]
filename = parts[-1]

img_path = image_lookup[(gcp_id, filename)]

img = cv2.imread(img_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

x = int(labels[sample]["mark"]["x"])
y = int(labels[sample]["mark"]["y"])



cv2.rectangle(
    img,
    (x-24,y-24),
    (x+24,y+24),
    (255,0,0),
    3
)

plt.figure(figsize=(10,10))
plt.imshow(img)
plt.show()

In [ ]:



# ============================================================
# CREATE DATASET.YAML
# ============================================================

yaml_content = f"""
path: {CONFIG["YOLO_ROOT"]}

train: images/train
val: images/val

names:
  0: Cross
  1: Square
  2: L-Shape
"""

yaml_path = os.path.join(
    CONFIG["YOLO_ROOT"],
    "dataset.yaml"
)

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"Saved: {yaml_path}")

In [ ]:
from ultralytics import YOLO

# ============================================================
# YOLO TRAINING
# ============================================================

model = YOLO(CONFIG["YOLO_MODEL"])


model.train(
    data=os.path.join(
        CONFIG["YOLO_ROOT"],
        "dataset.yaml"
    ),
    epochs=CONFIG["EPOCHS"],
    imgsz=CONFIG["IMAGE_SIZE"],
    batch=CONFIG["BATCH_SIZE"],
)

In [ ]:
# ============================================================
# INFERENCE
# ============================================================


from ultralytics import YOLO


CONFIG["CLASS_NAMES"] = {
    0: "Cross",
    1: "Square",
    2: "L-Shape"
}

# ============================================================
# LOAD MODEL
# ============================================================

print("Loading model...")

model = YOLO(
    CONFIG["MODEL_PATH"]
)

# ============================================================
# COLLECT TEST IMAGES
# ============================================================

test_images = []

for root, _, files in os.walk(
    CONFIG["TEST_ROOT"]
):

    for file_name in files:

        if file_name.lower().endswith(
            (".jpg", ".jpeg", ".png")
        ):

            full_path = os.path.join(
                root,
                file_name
            )

            relative_path = os.path.relpath(
                full_path,
                CONFIG["TEST_ROOT"]
            )

            test_images.append(
                (full_path, relative_path)
            )

print(
    f"Total Test Images: {len(test_images)}"
)

# ============================================================
# RUN INFERENCE
# ============================================================

predictions = {}

for full_path, relative_path in tqdm(
    test_images,
    desc="Running Inference"
):

    results = model(
        full_path,
        conf=CONFIG["CONF_THRESHOLD"],
        verbose=False
    )

    result = results[0]

    # --------------------------------------------------------
    # No detection
    # --------------------------------------------------------

    if len(result.boxes) == 0:

        predictions[relative_path] = {
            "mark": {
                "x": -1.0,
                "y": -1.0
            },
            "verified_shape": "Cross"
        }

        continue

    # --------------------------------------------------------
    # Highest confidence detection
    # --------------------------------------------------------

    best_idx = result.boxes.conf.argmax()

    box = result.boxes.xyxy[
        best_idx
    ].cpu().numpy()

    cls_id = int(
        result.boxes.cls[
            best_idx
        ]
    )

    x1, y1, x2, y2 = box

    center_x = float(
        (x1 + x2) / 2
    )

    center_y = float(
        (y1 + y2) / 2
    )

    predictions[relative_path] = {
        "mark": {
            "x": center_x,
            "y": center_y
        },
        "verified_shape": CONFIG[
            "CLASS_NAMES"
        ][cls_id]
    }

# ============================================================
# SAVE PREDICTIONS
# ============================================================

with open(
    CONFIG["PREDICTIONS_JSON"],
    "w"
) as f:

    json.dump(
        predictions,
        f,
        indent=4
    )

print(
    f"Saved: {CONFIG['PREDICTIONS_JSON']}"
)

print(
    f"Total predictions: {len(predictions)}"
)